# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arslan1Asim/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
userdata.get('HF_TOKEN')


from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOKEN"))

In [5]:
from google.colab import userdata

try:
    token = userdata.get("HF_TOKEN")
    print("Found:", token[:10] + "...")
except Exception as e:
    print(type(e).__name__, e)

Found: hf_EClfOUW...


In [6]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOKEN"))

In [10]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [9]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [11]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Unit of analysis + time window

**What one row means**

One row represents the daily performance of one content item for one client on a specific report date. The intended unique key is `client_hash_id`, `content_hash_id`, and `report_date`.

I verified this by checking the number of total rows against the number of unique client-content-date combinations. The dataset contains 78,835,655 total rows and 78,829,265 unique client-content-date combinations, resulting in 6,390 excess duplicate rows. I inspected a duplicate example and found that the duplicate rows were identical across all available fields. Therefore, the intended unit of analysis is one client-content-date observation, but exact duplicate rows should be removed before modeling.

**Which table(s) I'll use**

I will use the `fact_content_daily_performance` warehouse table.

**Which time window**

The full dataset spans 2025-01-27 to 2026-06-30 (520 report dates, 18 months), but row-generation behavior changes sharply around October 2025: from Jan–Sep 2025, GSC coverage is 100% and GA4/AI-session data is entirely absent, suggesting rows only existed when a GSC observation was present. Starting Oct 2025, monthly row volume jumps 3–4x, GSC coverage drops to 25–37% and never recovers, and GA4/AI-session data begins appearing. Because this is a structural break in how rows are generated rather than gradual data decay, I will restrict my development window to **November 2025 onward**, where row-generation is consistent, and treat the pre-October-2025 period as a separate cohort I won't train on.

**What I'll predict or rank**

I will use historical content performance information to predict or rank future content performance.

**One thing deliberately excluded**

Future performance information will be excluded from the feature set because it would not be available at prediction time and could cause data leakage.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql("""
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [15]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS unique_client_content_dates
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────────────┐
│ total_rows │ unique_client_content_dates │
│   int64    │            int64            │
├────────────┼─────────────────────────────┤
│   78835655 │                    78829265 │
└────────────┴─────────────────────────────┘

In [16]:
con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬───────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ row_count │
│         varchar         │         varchar          │    date     │   int64   │
├─────────────────────────┼──────────────────────────┼─────────────┼───────────┤
│ client_06d356715a8ff3b6 │ content_03a8b5950a52518a │ 2026-06-13  │         2 │
│ client_1a730cb2640a1abf │ content_6604767cde89152e │ 2026-06-13  │         2 │
│ client_1a730cb2640a1abf │ content_b5aec9a8a2ee7fb0 │ 2026-06-13  │         2 │
│ client_4a18d1793d92fb84 │ content_53707b72563016a4 │ 2026-06-14  │         2 │
│ client_1a8bf67cad4ee525 │ content_2630830d5f397c6c │ 2026-06-15  │         2 │
│ client_810019792c9b8efc │ content_26b4ce630106d689 │ 2026-06-16  │         2 │
│ client_06d356715a8ff3b6 │ content_f5a0c77c1826b467 │ 2026-06-16  │         2 │
│ client_1a730cb2640a1abf │ content_ac05941d77071556 │ 2026-06-18  │         2 │
│ client_b77d0d5f08f05e64 │ 

In [17]:
con.sql("""
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE client_hash_id = 'client_06d356715a8ff3b6'
  AND content_hash_id = 'content_03a8b5950a52518a'
  AND report_date = '2026-06-13'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

In [18]:
#time window

con.sql("""
SELECT
    MIN(report_date) AS earliest_report_date,
    MAX(report_date) AS latest_report_date,
    COUNT(DISTINCT report_date) AS number_of_report_dates,
    COUNT(DISTINCT month) AS number_of_months
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────┬────────────────────┬────────────────────────┬──────────────────┐
│ earliest_report_date │ latest_report_date │ number_of_report_dates │ number_of_months │
│         date         │        date        │         int64          │      int64       │
├──────────────────────┼────────────────────┼────────────────────────┼──────────────────┤
│ 2025-01-27           │ 2026-06-30         │                    520 │               18 │
└──────────────────────┴────────────────────┴────────────────────────┴──────────────────┘

## 2. Fields: feature / label / context / excluded

**Label:** `gsc_clicks` on a future report_date — the metric I'm predicting/ranking. I chose GSC
over GA4 for the label because ga4_data_available is false for ~96.4% of rows; a GA4-based label
would only be defined for a small, possibly unrepresentative slice of the dataset.

**Feature:** Lagged values of gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position,
ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec,
sessions_organic/direct/referral/social/paid/ai, ai_chatgpt/perplexity/gemini/copilot/claude/meta/other,
and scroll_events — all from dates prior to the prediction date. Using same-date values as features
would leak the label, since they're observed at the same time as what I'm predicting. GA4 and
AI-referral fields will be included as sparse/optional features rather than relied on, given their
low coverage.

**Context:** report_date and month situate each row in time and define the lag window; they aren't
fed to the model as raw features.

**Excluded:** client_hash_id and content_hash_id are identifiers with no generalizable predictive
signal (though they're needed to join/group rows correctly). client_has_gsc, client_has_ga4,
gsc_data_available, and ga4_data_available are availability flags rather than behavioral features.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(ds.features)

{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_meta': Value('int64'), 'ai_other': Value('int64'), 'scroll_events': Value('in

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Missing values: how often is each source actually populated?
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available THEN 0 ELSE 1 END) AS gsc_unavailable_rows,
    SUM(CASE WHEN ga4_data_available THEN 0 ELSE 1 END) AS ga4_unavailable_rows,
    SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS null_avg_position
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
""")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┬──────────────────────┬───────────────────┐
│ total_rows │ gsc_unavailable_rows │ ga4_unavailable_rows │ null_avg_position │
│   int64    │        int128        │        int128        │      int128       │
├────────────┼──────────────────────┼──────────────────────┼───────────────────┤
│   78835655 │             49865604 │             76019200 │          49865654 │
└────────────┴──────────────────────┴──────────────────────┴───────────────────┘

In [20]:
# Window check: does data availability shift over time? (e.g. AI referral columns rolling out late)
con.sql("""
SELECT
    month,
    COUNT(*) AS rows,
    SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS ga4_available_rows,
    SUM(sessions_ai) AS total_ai_sessions
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
GROUP BY month
ORDER BY month
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬──────────┬────────────────────┬────────────────────┬───────────────────┐
│  month  │   rows   │ gsc_available_rows │ ga4_available_rows │ total_ai_sessions │
│ varchar │  int64   │       int128       │       int128       │      int128       │
├─────────┼──────────┼────────────────────┼────────────────────┼───────────────────┤
│ 2025-01 │     1297 │               1297 │                  0 │                 0 │
│ 2025-02 │    75985 │              75985 │                  0 │                 0 │
│ 2025-03 │   167859 │             167859 │                  0 │                 0 │
│ 2025-04 │   285114 │             285114 │                  0 │                 0 │
│ 2025-05 │   349923 │             349923 │                  0 │                 0 │
│ 2025-06 │   329201 │             329201 │                  0 │                 0 │
│ 2025-07 │   469794 │             469794 │                  0 │                 0 │
│ 2025-08 │   704962 │             704962 │                  0 │ 

In [21]:
# Window check: does every client have data across the full window, or do clients enter/exit at different times?
con.sql("""
SELECT
    client_hash_id,
    MIN(report_date) AS first_seen,
    MAX(report_date) AS last_seen,
    COUNT(DISTINCT report_date) AS days_present
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
GROUP BY client_hash_id
ORDER BY days_present ASC
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬────────────┬────────────┬──────────────┐
│     client_hash_id      │ first_seen │ last_seen  │ days_present │
│         varchar         │    date    │    date    │    int64     │
├─────────────────────────┼────────────┼────────────┼──────────────┤
│ client_aef6ffea193da149 │ 2026-05-28 │ 2026-06-30 │           34 │
│ client_04660893ae39614a │ 2026-05-22 │ 2026-06-30 │           40 │
│ client_a22068e339bf95f5 │ 2026-05-22 │ 2026-06-30 │           40 │
│ client_c353557474475e51 │ 2026-04-30 │ 2026-06-30 │           52 │
│ client_1a8bf67cad4ee525 │ 2026-05-09 │ 2026-06-30 │           53 │
│ client_c7c2962f1c9c3089 │ 2026-05-09 │ 2026-06-30 │           53 │
│ client_7de9989c909e91a5 │ 2026-04-20 │ 2026-06-30 │           72 │
│ client_9c26c096d6e57253 │ 2026-04-16 │ 2026-06-30 │           76 │
│ client_0b245132bb722950 │ 2026-04-06 │ 2026-06-30 │           86 │
│ client_8ddc46da5414ffd8 │ 2026-04-02 │ 2026-06-30 │           90 │
│ client_06d356715a8ff3b6 │ 2026-0


## 4. Data limits

- **GA4 coverage is very sparse**: ga4_data_available is false for 76,019,200 of 78,835,655 rows
  (~96.4%). Any feature or label built from ga4_* fields would only apply to a small, possibly
  unrepresentative slice of the dataset — most rows simply have no analytics signal to draw on.

- **GSC coverage is also incomplete**: gsc_data_available is false for 49,865,604 rows (~63.2%).
  Combined with the GA4 gap, a large share of client-content-date observations have partial or no
  behavioral data at all.

- **The row-generation logic changed around October 2025**: from Jan–Sep 2025, GSC coverage is
  100% and GA4/AI-session data is entirely absent (0 rows) — suggesting rows only existed when a
  GSC observation was present. Starting Oct 2025, monthly row volume jumps 3–4x and GSC coverage
  drops to 25–37% and never returns to 100%, while GA4 and AI-session data begin appearing (still
  under ~6% and ~0.3% coverage respectively). This is a structural break, not gradual decay —
  coverage rates and row counts from before and after Oct 2025 aren't directly comparable, and a
  model spanning this boundary risks learning the schema change rather than real behavior. I will
  restrict my development window to Nov 2025 onward, where row-generation is consistent, or treat
  the pre-Oct-2025 period as a separate cohort.

- **NULL avg_position doesn't fully track the availability flag**: gsc_avg_position is NULL for
  49,865,654 rows — 50 more than the rows flagged gsc_data_available = false, likely rows with 0
  impressions where average position is undefined even though the row is otherwise "available."

- **Client tenure mixes onboarding and churn**: most short-history clients simply started recently —
  first_seen falls anywhere from March–June 2026, but last_seen is uniformly 2026-06-30, meaning
  they're still active at the end of the window. A separate small cluster of clients (e.g. those
  starting 2025-11-05) stopped appearing by February 2026, well before the window ends — these
  look like churned clients rather than late onboards. days_present alone doesn't distinguish
  "new" from "gone," so any per-client history length needs to be checked against last_seen
  relative to the window's end before being used as a feature or for splitting train/test data.

- **This data is directional, not causal**: it shows what happened, not why — it can't separate a
  ranking change caused by content quality from one caused by algorithm updates or seasonality.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Directly verify the "50 anomalous rows" claim: gsc_data_available=true but avg_position still NULL
con.sql("""
SELECT COUNT(*) AS anomalous_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE gsc_data_available = true AND gsc_avg_position IS NULL
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ anomalous_rows │
│     int64      │
├────────────────┤
│             50 │
└────────────────┘

In [23]:
# Directly verify the churn claim: how many clients' last_seen falls well before the window end (2026-06-30)?
con.sql("""
SELECT
    COUNT(*) AS churned_clients
FROM (
    SELECT client_hash_id, MAX(report_date) AS last_seen
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    GROUP BY client_hash_id
) t
WHERE last_seen < DATE '2026-06-30' - INTERVAL 30 DAY
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┐
│ churned_clients │
│      int64      │
├─────────────────┤
│               4 │
└─────────────────┘

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.